Variables are
labels, not boxes.

In [1]:
a = [1,2,3,4]
b = a
a.append(5)
b

[1, 2, 3, 4, 5]

In [2]:
class Gizmo:
    def __init__(self):
        print(f'Gizmo id: {id(self)}')


x = Gizmo()

y = Gizmo() * 10

Gizmo id: 2683904752384
Gizmo id: 2683903537952


TypeError: unsupported operand type(s) for *: 'Gizmo' and 'int'

Python evaluates the right side:

After the object exists, Python binds the name y to it. (this why it was printed)

then it tries to multiply and raise an error

"object creation happens BEFORE variable binding"

### object identity and object equality

An object’s identity never changes once it has been created

In [5]:
charles = {'name': 'Charles L. Dodgson', 'born': 1832, 'balance': 950}
lewis = charles
alex = {'name': 'Charles L. Dodgson', 'born': 1832, 'balance': 950}

lewis == alex, lewis is alex, lewis is charles

(True, False, True)

Why None should use is

None is a singleton in Python.

That means there is only one None object in the whole program.

In [ ]:
class Node:
    def __init__(self, value, left=None, right=None):
        self.value = value
        self.left = left
        self.right = right

END_OF_DATA = object()

def traverse(node):

    if node is END_OF_DATA:  # check sentinel objects
        return

    if node is None:
        return

    print(node.value)

    traverse(node.left)
    traverse(node.right)


traverse(Node(
    "A",
    Node("B"),
    Node("C")
))


A
B
C


Use == for value comparison, but use is when checking singletons like None or sentinel objects, because you want to know if two variables refer to the exact same object.

In [ ]:
# Tuple are hashable if all items inside are
a = (5,[1,2])
b = (5,[1,2])
a == b  #True
id(b[-1])


2683904823936

In [14]:
b[-1].append(3)
a == b #False
id(b[-1])

2683904823936

In [57]:
# Tuples are hashable only if all elements are hashable.
hash(a)

NameError: name 'a' is not defined

In [ ]:
t = (1,2)
t_id = id(t)

t += (3,)
print(t)      # (1,2,3)
t_id,id(t)  # different id!

(1, 2, 3)


(2683920030208, 2683919758784)

### copies are shallow


In [ ]:
list1 = [5,[5,4,3],(3,2,1)]
list2 = list1[:]  #or list(list1)
list2, list1==list2, list1 is list2

([5, [5, 4, 3], (3, 2, 1)], True, False)

In [ ]:
list1.append(100)
list1,list2,list1==list2
list1.remove(100)

In [ ]:
list1[1].append(99)
list1,list2,list1==list2

([5, [5, 4, 3, 99], (3, 2, 1)], [5, [5, 4, 3, 99], (3, 2, 1)], True)

In [ ]:
list1[2] += (5,) #this create a new tuple, tuples are unmutable
list1,list2

([5, [5, 4, 3, 99], (3, 2, 1, 5)], [5, [5, 4, 3, 99], (3, 2, 1)])

Deep copies: duplicates that do not share references of embedded objects)

In [ ]:
import copy
a = [10, 20]
b = copy.copy(a)
c = copy.deepcopy(a)
a.append(100)
a,b,c


([10, 20, 100], [10, 20], [10, 20])

shallow copy appears like deep copy if the list contains only immutable objects.

In [ ]:
a = [10, [20, 30]]
b = copy.copy(a)       # shallow copy
c = copy.deepcopy(a)   # deep copy
a[1].append(99)
a,b,c

([10, [20, 30, 99]], [10, [20, 30, 99]], [10, [20, 30]])

In [ ]:
class ClassRoom:
    def __init__(self, students=None):
        if students is None:
            self.students = []
        else:
            self.students = list(students)  # NO direct assignment (=student) , but make a shallow copy

students_list = ['Mike', 'Pao']
room = ClassRoom(students_list)

room.students.append('Lucy')
print(students_list)  # ['Mike', 'Pao']  ← unchanged
print(room.students)  # ['Mike', 'Pao', 'Lucy']  ← class list changed

['Mike', 'Pao']
['Mike', 'Pao', 'Lucy']


In [ ]:
st = copy.deepcopy(room.students) # i did it like that by accessing an attribute, but a good practice is add an append method within the class
st.append('moha')
st,room.students


(['Mike', 'Pao', 'Lucy', 'moha'], ['Mike', 'Pao', 'Lucy'])

In [ ]:
# Mutable Types as Parameter Defaults: Bad Idea
class ClassRoom:
    def __init__(self, students=[]):  # mutable default
        self.students = students

room1 = ClassRoom()
room1.students.append('Mike')

room2 = ClassRoom()  # we did not pass students
print(room2.students) 

['Mike']


In [ ]:
room1 = ClassRoom(['Mike', 'Pao'])  # passed its own list
room2 = ClassRoom()                  # uses default []
room3 = ClassRoom()                  # uses default []

room2.students.append('Alice')
room3.students.append('Bob')

print(room2.students)  # ['Alice', 'Bob']
print(room3.students)  # ['Alice', 'Bob']
print(room1.students)  # ['Mike', 'Pao'] → independent

['Mike', 'Alice', 'Bob']
['Mike', 'Alice', 'Bob']
['Mike', 'Pao']


In [ ]:
ClassRoom.__init__.__defaults__
# explains why None is commonly used as the default
#value for parameters that may receive mutable values

(['Mike', 'Alice', 'Bob'],)

Always assign to self.attribute carefully:

If the argument is mutable and optional, create a new object inside __init__ to avoid shared references.

Direct assignment (self.attribute = attribute) is safe only if the caller owns the object exclusively and you are okay with sharing it.

| **Case / Type**                                    | **Example**                                                       | **Do you assign to `self`?** | **Comment / Why**                                                                                     |
| -------------------------------------------------- | ----------------------------------------------------------------- | ---------------------------- | ----------------------------------------------------------------------------------------------------- |
| **Immutable argument (number, string, tuple)**     | `x = 10` → `self.x = x`                                           | ✅ Yes                        | Safe to assign; each object keeps its own value.                                                      |
| **Mutable argument passed by caller (list, dict)** | `students = ['Alice', 'Bob']` → `self.students = list(students)`  | ✅ Usually (make a copy)      | Assigning directly shares the same object; use copy to avoid side effects.                            |
| **Mutable default in constructor**                 | `students=[]` → `self.students = []` inside `if students is None` | ✅ Always                     | Never assign the default directly; all objects would share the same list.                             |
| **Temporary calculation inside `__init__`**        | `area = width * height`                                           | ⚠ Optional                   | Assign to `self.area = width*height` **if you want object to remember it**, otherwise leave as local. |
| **Local variable only used in a method**           | `result = x + y` inside `compute()`                               | ❌ No                         | Temporary variable; disappears when method ends.                                                      |
| **Class-level constant / shared value**            | `pi = 3.14159` defined in class body                              | ❌ No (`Circle.pi`)           | Shared by all instances; not stored per object.                                                       |
| **Nested mutable object in argument**              | `matrix = [[1,2],[3,4]]` → `self.matrix = copy.deepcopy(matrix)`  | ✅ Yes (deep copy)            | Needed to avoid shared references to inner lists.                                                     |


Assign to self if you want the object to “remember” it later.

Do not assign if it’s temporary, local to the method, or shared as class constant.

Mutable objects → make a copy unless intentional sharing is okay.

del deletes references, not objects. Python’s
garbage collector may discard an object from memory as an indirect result of del, if
the deleted variable was the last reference to the object

In [ ]:
a = [1,2,3]
b = a 
del a
print('a' in globals())
print(b) # u need to del b to be discarder from garbage

False
[1, 2, 3]


In [62]:
t1 = (1,2)
t2 = (1,2)
t1 is t2, t1 == t2

(False, True)

In [64]:
s1 = "aa"
s2 = "aa"
s1 is s2, s1 == s2


(True, True)

The sharing of string literals is an optimization technique called interning. CPython
uses a similar technique with small integers to avoid unnecessary duplication of num‐
bers that appear frequently in programs like 0, 1, –1, etc. Note that CPython does not
intern all strings or integers, and the criteria it uses to do so is an undocumented
implementation detail.